# Immigration Guidance RAG System - Phase 1 (Ollama)

This notebook implements a **Retrieval-Augmented Generation (RAG)** system for answering questions about USCIS immigration policies using official documentation.

## Project Overview

**Goal**: Build an intelligent assistant that can answer immigration policy questions with accurate, source-cited responses.

**Tech Stack**:
- **LLM**: Llama3 (via Ollama - runs locally, no API costs)
- **Embeddings**: HuggingFace sentence-transformers/all-MiniLM-L6-v2 (free, local)
- **Vector Database**: Chroma (persistent storage)
- **Framework**: LangChain for orchestration

**Why RAG?**
- LLMs alone have knowledge cutoffs and can hallucinate
- RAG grounds responses in actual source documents
- Provides verifiable citations for legal/policy questions

**Data**: 47 USCIS Policy Manual PDF documents (~319 pages total)

---

## Step 1: Setup and Dependencies

Initialize the required libraries and configure Ollama connection.

**Note**: This implementation uses Ollama for local LLM inference. Make sure Ollama is running locally (`ollama serve`) before executing these cells.

In [1]:
import langchain
import requests
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "Ollama3"
model = Ollama(model=MODEL)
parser = StrOutputParser()
chain = model | parser


a:\rag-immigration-guidance\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aaron fraser\AppData\Local\Temp\ipykernel_25772\901718329.py:12: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  model = Ollama(model=MODEL)


In [2]:

def ask_ollama(prompt, model="llama2"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()["response"]

## Step 2: Document Loading

Load all PDF documents from the `data/raw/` directory using LangChain's PyPDFLoader.

**Process**:
1. Find all PDF files in the data directory
2. Load each PDF page-by-page
3. Extract text content and metadata (source file, page number)

**Expected Output**: ~203 pages from 25 USCIS policy documents

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

pdf_dir = Path(r"A:\rag-immigration-guidance\data\raw")
pdf_files = list(pdf_dir.glob("*.pdf"))

documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

print("Total pages loaded:", len(documents))


Total pages loaded: 319


## Step 3: Text Chunking

Split documents into smaller chunks for more precise retrieval.

**Why chunk?** 
- Full pages are too large for effective embedding/retrieval
- Smaller chunks allow finding specific relevant sections
- Balance: too small = loss of context, too large = imprecise retrieval

**Parameters Chosen**:
- `chunk_size=1000`: Each chunk is ~1000 characters (~250 words)
- `chunk_overlap=150`: 150-character overlap prevents splitting mid-context
- **Separators**: Prioritize splitting on paragraphs → sentences → words

This creates more targeted, retrievable pieces of information.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)


split_docs = text_splitter.split_documents(documents)

## Step 4: Create Embeddings and Vector Database

Convert text chunks into vector embeddings and store them in a searchable database.

**Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2`
- Runs locally (no API costs)
- Fast inference on CPU
- Good quality for semantic search
- Outputs 384-dimensional vectors

**Vector Database**: Chroma
- Stores embeddings with metadata (source, page number)
- Enables fast similarity search
- **Persists to disk** (`./chroma_db`) for reuse across sessions

**Note**: First run may take 1-2 minutes to embed all chunks. Subsequent runs load instantly from disk.

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"  
)

C:\Users\aaron fraser\AppData\Local\Temp\ipykernel_25772\4034914147.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2000.13it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 5: Initialize Local LLM (Ollama)

Set up Llama3 via Ollama for text generation.

**Why Llama3 + Ollama?**
- 100% free and runs locally
- Good reasoning capabilities
- No API rate limits or costs
- Privacy: data never leaves your machine

**Configuration**:
- `temperature=0`: Deterministic, factual responses (good for legal/policy questions)
- Model: `llama3` (8B parameter version recommended for speed/quality balance)

**Prerequisite**: Make sure Ollama is running (`ollama serve` in terminal)

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage


llm = ChatOllama(
    model="llama3",
    temperature=0
)


## Step 6: Build the RAG Pipeline

Combine retrieval and generation into a complete RAG chain using LangChain Expression Language (LCEL).

**Components**:
1. **Prompt Template**: Instructs the LLM to act as an immigration expert and cite sources
2. **Retriever**: Finds the top `k=4` most relevant chunks for each question
3. **RAG Chain**: Orchestrates: Question → Retrieve Context → Format → Generate Answer

**How it works**:
```
User Question 
    ↓
Vector Search (find top 4 similar chunks)
    ↓
Format chunks as context
    ↓
Prompt: Context + Question → LLM
    ↓
Generated Answer with Citations
```

This ensures answers are grounded in actual source documents, not hallucinated.

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt_template = """You are an expert assistant specializing in USCIS policy guidance. 
Use the following context from official USCIS documents to answer the question provided.

Important: 
- Cite specific sources by referencing the document chapter and page numbers
- If the context doesn't contain enough information to fully answer the question, acknowledge this
- Provide clear, actionable guidance when applicable
- Use precise legal terminology from the source documents

Context:
{context}

Question: {question}

Answer:"""
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | PROMPT
    | llm
    | StrOutputParser()
)

## Step 7: Query Function

Helper function to query the RAG system and display formatted results with source attribution.

**Features**:
- Retrieves relevant source documents
- Generates answer using RAG chain
- Displays both answer and sources (document name + page number)
- Returns structured result for further processing

In [8]:
def ask_question(question: str):
    """
    Query the RAG system and display results with sources.
    """
    print(f"\n{'='*80}")
    print(f"QUESTION: {question}")
    print(f"{'='*80}\n")
    

    source_docs = retriever.invoke(question)
    

    answer = rag_chain.invoke(question)
    

    print("ANSWER:")
    print(answer)
    
    print(f"\n{'─'*80}")
    print("SOURCES:")
    for i, doc in enumerate(source_docs, 1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")
        print(f"  [{i}] {Path(source).name} - Page {page}")
    print(f"{'='*80}\n")
    
    return {"result": answer, "source_documents": source_docs}

## Step 8: Test the System

Now you can ask questions about immigration policies! 

**Example questions to try**:
- "What are the requirements for U.S. citizenship?"
- "What is the Child Status Protection Act?"
- "What documentation is required for naturalization?"
- "What happens during an immigration interview?"

Uncomment the cell below and add your own questions:

In [9]:
result1= ask_question("What are the key requirements for U.S. citizenship?")


QUESTION: What are the key requirements for U.S. citizenship?

ANSWER:
According to the USCIS Policy Manual, Chapter 2 - Becoming a U.S. Citizen, the key requirements for U.S. citizenship are not explicitly stated in the provided context. However, we can infer that the applicant must fulfill all of the requirements established by law, regardless of whether they are naturalized through a formal application process or by operation of law (deriving citizenship).

To provide more specific guidance, I would recommend consulting the relevant sections of the USCIS Policy Manual, such as Volume 12, Part A, Chapter 2, which provides detailed information on the requirements for U.S. citizenship.

In particular, Section 2.1(a) of the Policy Manual states that "to be eligible for naturalization, an applicant must meet the following requirements: (1) be at least 18 years old; (2) have been a permanent resident for at least 5 years or have been married to a U.S. citizen and lived in the United Stat

---

## Summary & Next Steps

### What We Built
✅ Complete RAG pipeline with local LLM (Llama3)  
✅ Document loading and intelligent chunking  
✅ Vector embeddings and persistent storage  
✅ Source-attributed answers from USCIS policies  

### Performance Characteristics
- **Cost**: $0 (completely free, runs locally)
- **Speed**: ~3-5 seconds per query (depends on hardware)
- **Quality**: Good for general questions, may struggle with complex legal reasoning
- **Privacy**: All data stays on your machine

### Comparison: Phase 1 (Ollama) vs Claude Implementation

| Aspect | Phase 1 (This Notebook) | Claude Version |
|--------|------------------------|----------------|
| LLM | Llama3 (local) | Claude Sonnet 4.5 (API) |
| Cost | Free | ~$0.01-0.02/query |
| Setup | Requires Ollama running | Just API key |
| Quality | Good | Excellent (superior legal reasoning) |
| Speed | 3-5s (local) | 2-3s (API) |
| Privacy | Full | Data sent to Anthropic |

### Possible Improvements
- Add conversation memory for follow-up questions
- Implement query rewriting for better retrieval
- Add evaluation metrics to measure answer quality
- Build a web interface (see `app.py` for Streamlit implementation)
- Fine-tune retrieval parameters (chunk size, k value, similarity threshold)